# CPE 342 Term Project — Task 3: Player Monthly Spending Prediction (Regression)
**Course:** CPE 342 Machine Learning (Semester 1/2026) — KMUTT  
**System Component:** Karena Thailand Player Intelligence System — Monetization & Revenue Forecasting Module  
**Task Category:** Zero-Inflated Continuous Regression (`spending_30d` in THB $\ge 0$)  
**Primary Evaluation Metric:** **Normalized MAE Score** ($\text{NMAE} = \frac{1}{1 + \sum|y - \hat{y}| / \sum y} \in [0, 1]$, higher is better)  
**Competition Rule Compliance:**
* **Data Usage Policy (Rule 2):** Uses **exclusively** `task3/data/train.csv` for training and validation (`test.csv` is strictly isolated for final inference only; zero test-set pseudo-labeling).
* **Model Development (Rule 3):** Two-Stage Hurdle & L1 Regressors trained **100% from scratch** on provided data. Zero AutoML frameworks; hyperparameter tuning via `Optuna` is supported.
* **Portable Execution:** Supports **Google Colab GPU**, **Kaggle Kernels**, and **Local Workspace (`uv`)**

## 1. Environment Setup & Path Resolution

In [ ]:
import os
import sys
import warnings
from pathlib import Path
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore")

for pkg in ["lightgbm", "xgboost", "catboost", "optuna"]:
    try:
        __import__(pkg)
    except ImportError:
        if "google.colab" in sys.modules or "KAGGLE_KERNEL_RUN_TYPE" in os.environ:
            os.system(f"pip install -q {pkg}")

import lightgbm as lgb
import xgboost as xgb
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import mean_absolute_error, roc_auc_score

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

BASE_DIR = Path.cwd()
if (BASE_DIR / "project" / "task3").exists():
    TASK_DIR = BASE_DIR / "project" / "task3"
elif (BASE_DIR / "task3").exists():
    TASK_DIR = BASE_DIR / "task3"
else:
    TASK_DIR = BASE_DIR

DATA_DIR = TASK_DIR / "data"
MODELS_DIR = TASK_DIR / "models"
PREDICTIONS_DIR = TASK_DIR / "predictions"
ASSETS_DIR = TASK_DIR / "assets"

for d in [DATA_DIR, MODELS_DIR, PREDICTIONS_DIR, ASSETS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

plt.rcParams["figure.dpi"] = 120
plt.rcParams["savefig.dpi"] = 300
plt.rcParams["font.sans-serif"] = ["Sarabun", "TH Sarabun New", "DejaVu Sans", "Arial"]
plt.rcParams["axes.unicode_minus"] = False

def nmae_score(y_true, y_pred):
    y_pred_clip = np.clip(y_pred, 0.0, None)
    rel_mae = np.sum(np.abs(y_true - y_pred_clip)) / (np.sum(y_true) + 1e-9)
    return float(1.0 / (1.0 + rel_mae))

print(f"Task 3 Directory: {TASK_DIR}")

## 2. Data Loading & Zero-Inflated Target Distribution Analysis
จากการตรวจสอบ `task3/data/train.csv` ($104,000$ แถว $\times 34$ คอลัมน์):
* ผู้เล่นจำนวน **50,049 คน (48.12%) มียอดใช้จ่าย `spending_30d == 0.0` THB** (Zero-Inflated Distribution)
* ส่วนผู้เล่นที่มียอดใช้จ่าย ($> 0$ THB จำนวน 53,951 คน) มีการกระจายตัวเบ้ขวาอย่างรุนแรง (Heavy Right Skew: ค่ามัธยฐานรวม 79.44 บาท, ค่าเฉลี่ย 10,497.04 บาท, สูงสุด 300,000.00 บาท)

In [ ]:
train_df = pd.read_csv(DATA_DIR / "train.csv")
test_df = pd.read_csv(DATA_DIR / "test.csv")

ID_COL = "player_id"
TARGET_COL = "spending_30d"
RAW_FEATURES = [c for c in test_df.columns if c != ID_COL]

y_train = train_df[TARGET_COL].values
is_spender = (y_train > 0).astype(int)

print(f"Train Shape       : {train_df.shape} | Test Shape: {test_df.shape}")
print(f"Zero Spenders     : {(y_train == 0).sum():,} ({(y_train == 0).mean()*100:.2f}%)")
print(f"Positive Spenders : {(y_train > 0).sum():,} ({(y_train > 0).mean()*100:.2f}%)")
print(f"Positive Spenders Summary:\n{pd.Series(y_train[y_train > 0]).describe().round(2)}")

fig, axes = plt.subplots(1, 2, figsize=(13.5, 5))
axes[0].bar(["Non-Spender (0 THB)", "Positive Spender (> 0 THB)"],
            [(y_train == 0).sum(), (y_train > 0).sum()],
            color=["#64748B", "#16A34A"], width=0.55)
axes[0].set_title("Task 3: Zero-Inflated Spending Split (N = 104,000)", fontsize=12, fontweight="bold")
axes[0].set_ylabel("Number of Players")
axes[0].grid(axis="y", alpha=0.3)

axes[1].hist(np.log1p(y_train[y_train > 0]), bins=50, color="#2563EB", edgecolor="white", alpha=0.85)
axes[1].set_title("Log-Transformed Spending Distribution log(1 + THB) for Spenders", fontsize=12, fontweight="bold")
axes[1].set_xlabel("log(1 + spending_30d)")
axes[1].set_ylabel("Frequency")
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig(ASSETS_DIR / "zero-inflated-spending-distribution.png", dpi=300, bbox_inches="tight")
plt.show()

## 3. Financial & RFM (Recency-Frequency-Monetary) Feature Engineering
สร้างฟีเจอร์ตามหลัก **RFM Analysis** ของวงการเกม:
* **Monetary Velocity:** อัตราส่วนยอดใช้จ่ายเดือนก่อนหน้าเทียบกับค่าเฉลี่ยในอดีต (`prev_to_historical_ratio`), มูลค่าธุรกรรมรวมคาดการณ์ (`expected_tx_volume`)
* **Recency & Retention Decay:** อัตราส่วนความถี่การซื้อหารด้วยจำนวนวันนับจากการซื้อครั้งล่าสุด (`purchase_recency_decay`)
* **Segment & VIP Synergy:** ปฏิสัมพันธ์ระหว่าง `segment` (จาก Task 2), `vip_status`, และ `historical_spending`

In [ ]:
def engineer_task3_features(df: pd.DataFrame) -> pd.DataFrame:
    out = df[RAW_FEATURES].copy()
    eps = 1e-5
    
    out["missing_count"] = out[RAW_FEATURES].isna().sum(axis=1)
    
    # 1. Monetary & Historical Spending Dynamics
    out["log_historical_spending"] = np.log1p(out["historical_spending"].clip(lower=0))
    out["log_prev_month_spending"] = np.log1p(out["prev_month_spending"].clip(lower=0))
    out["log_avg_tx_value"] = np.log1p(out["avg_transaction_value"].clip(lower=0))
    out["prev_to_hist_ratio"] = out["prev_month_spending"] / (out["historical_spending"] + 1.0)
    out["est_lifetime_tx_spend"] = out["total_transactions"] * out["avg_transaction_value"]
    out["monthly_historical_rate"] = (out["historical_spending"] * 30.0) / (out["account_age_days"] + 30.0)
    out["recent_momentum_diff"] = out["prev_month_spending"] - out["monthly_historical_rate"]
    
    # 2. Recency & Frequency Ratios
    out["purchase_recency_score"] = 1.0 / (out["days_since_last_purchase"].clip(lower=0) + 1.0)
    out["login_recency_score"] = 1.0 / (out["days_since_last_login"].clip(lower=0) + 1.0)
    out["freq_to_recency_ratio"] = out["purchase_frequency"] / (out["days_since_last_purchase"].clip(lower=0) + 1.0)
    out["discount_purchase_share"] = out["purchases_on_discount"] / (out["total_transactions"] + 1.0)
    
    # 3. VIP, Segment & Engagement Interactions
    out["vip_x_prev_spend"] = (out["vip_status"].fillna(0) + 1.0) * out["prev_month_spending"]
    out["segment_x_prev_spend"] = (out["segment"].fillna(0) + 1.0) * out["prev_month_spending"]
    out["weekly_play_minutes"] = out["avg_session_length"] * out["sessions_per_week"]
    
    return out

X_train_fe = engineer_task3_features(train_df)
X_test_fe = engineer_task3_features(test_df)
FEATURE_COLS = list(X_train_fe.columns)
print(f"Engineered Feature Matrix — Train: {X_train_fe.shape}, Test: {X_test_fe.shape}")

## 4. Two-Stage Hurdle Model + Direct L1 (MAE) & Log-L1 Ensemble
เพื่อเพิ่มค่า **NMAE Score** ให้สูงสุด เราสร้างและเปรียบเทียบ 3 สถาปัตยกรรมหลักบน 5-Fold CV:
1. **Stage 1 Hurdle Classifier:** ทำนายความน่าจะเป็นที่ผู้เล่นจะมียอดใช้จ่าย $P(y_i > 0)$
2. **Stage 2A Positive Log-L1 Regressor:** ฝึกสอนด้วย `regression_l1` บน $\log(1 + y_i)$ เฉพาะผู้เล่นที่ $y_i > 0$
3. **Stage 2B Full-Data Direct L1 & Tweedie Regressors:** ฝึกสอนด้วย L1 Loss โดยตรงเพื่อลด $|\hat{y}_i - y_i|$
4. **Hurdle Post-Processing & Gating Optimization:** ตัดผู้เล่นที่มี $P(y_i > 0) < \tau_{\text{gate}}$ ให้เป็น `0.0 THB` และผสมผลทำนายในสัดส่วนที่ให้ค่า NMAE Score สูงที่สุด

In [ ]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

oof_prob_spend = np.zeros(len(train_df))
test_prob_spend = np.zeros(len(test_df))

oof_reg_log = np.zeros(len(train_df))
test_reg_log = np.zeros(len(test_df))

oof_reg_raw_l1 = np.zeros(len(train_df))
test_reg_raw_l1 = np.zeros(len(test_df))

feature_importances = np.zeros(len(FEATURE_COLS))

for fold, (trn_idx, val_idx) in enumerate(skf.split(X_train_fe, is_spender), 1):
    X_tr, X_va = X_train_fe.iloc[trn_idx], X_train_fe.iloc[val_idx]
    y_tr, y_va = y_train[trn_idx], y_train[val_idx]
    s_tr, s_va = is_spender[trn_idx], is_spender[val_idx]
    
    # Stage 1: Binary Hurdle Classifier P(spending_30d > 0)
    clf = lgb.LGBMClassifier(
        n_estimators=1200,
        learning_rate=0.03,
        num_leaves=63,
        subsample=0.8,
        colsample_bytree=0.8,
        min_child_samples=40,
        random_state=SEED + fold,
        n_jobs=-1,
        verbose=-1
    )
    clf.fit(X_tr, s_tr, eval_set=[(X_va, s_va)], callbacks=[lgb.early_stopping(80, verbose=False)])
    oof_prob_spend[val_idx] = clf.predict_proba(X_va)[:, 1]
    test_prob_spend += clf.predict_proba(X_test_fe)[:, 1] / skf.n_splits
    
    # Stage 2A: Log-L1 Regressor trained on Positive Spenders (y > 0)
    pos_mask_tr = y_tr > 0
    pos_mask_va = y_va > 0
    reg_log = lgb.LGBMRegressor(
        objective="regression_l1",
        n_estimators=1500,
        learning_rate=0.03,
        num_leaves=63,
        subsample=0.8,
        colsample_bytree=0.8,
        min_child_samples=30,
        reg_alpha=1.0,
        reg_lambda=2.0,
        random_state=SEED + fold,
        n_jobs=-1,
        verbose=-1
    )
    reg_log.fit(
        X_tr[pos_mask_tr], np.log1p(y_tr[pos_mask_tr]),
        eval_set=[(X_va[pos_mask_va], np.log1p(y_va[pos_mask_va]))],
        callbacks=[lgb.early_stopping(80, verbose=False)]
    )
    oof_reg_log[val_idx] = np.expm1(np.clip(reg_log.predict(X_va), 0, 15))
    test_reg_log += np.expm1(np.clip(reg_log.predict(X_test_fe), 0, 15)) / skf.n_splits
    feature_importances += reg_log.feature_importances_ / skf.n_splits
    
    # Stage 2B: Direct L1 Regressor on Full Cohort
    reg_raw = lgb.LGBMRegressor(
        objective="regression_l1",
        n_estimators=1500,
        learning_rate=0.03,
        num_leaves=63,
        subsample=0.8,
        colsample_bytree=0.8,
        min_child_samples=40,
        reg_alpha=1.0,
        reg_lambda=2.0,
        random_state=SEED + 100 + fold,
        n_jobs=-1,
        verbose=-1
    )
    reg_raw.fit(
        X_tr, y_tr,
        eval_set=[(X_va, y_va)],
        callbacks=[lgb.early_stopping(80, verbose=False)]
    )
    oof_reg_raw_l1[val_idx] = np.clip(reg_raw.predict(X_va), 0, 300000)
    test_reg_raw_l1 += np.clip(reg_raw.predict(X_test_fe), 0, 300000) / skf.n_splits
    
    print(f"Fold {fold}/5 | Hurdle AUC: {roc_auc_score(s_va, oof_prob_spend[val_idx]):.4f} | Raw L1 NMAE: {nmae_score(y_va, oof_reg_raw_l1[val_idx]):.4f}")

# Optimize Hurdle Gating & Blending on OOF to maximize NMAE Score
best_nmae = -1.0
best_params = (0.5, 0.3, 1.0)
for alpha in np.linspace(0.0, 1.0, 21):
    base_blend = alpha * (oof_prob_spend * oof_reg_log) + (1.0 - alpha) * oof_reg_raw_l1
    for gate_t in np.linspace(0.10, 0.50, 17):
        gated = np.where(oof_prob_spend < gate_t, 0.0, base_blend)
        for scale in np.linspace(0.85, 1.10, 26):
            sc = nmae_score(y_train, gated * scale)
            if sc > best_nmae:
                best_nmae = sc
                best_params = (float(alpha), float(gate_t), float(scale))

alpha_opt, gate_opt, scale_opt = best_params
oof_hurdle_final = np.where(
    oof_prob_spend < gate_opt,
    0.0,
    (alpha_opt * (oof_prob_spend * oof_reg_log) + (1.0 - alpha_opt) * oof_reg_raw_l1) * scale_opt
)
test_hurdle_final = np.where(
    test_prob_spend < gate_opt,
    0.0,
    (alpha_opt * (test_prob_spend * test_reg_log) + (1.0 - alpha_opt) * test_reg_raw_l1) * scale_opt
)

print(f"\nOOF NMAE (Median Baseline)        : {nmae_score(y_train, np.full_like(y_train, np.median(y_train))):.4f}")
print(f"OOF NMAE (Direct L1 GBDT)         : {nmae_score(y_train, oof_reg_raw_l1):.4f}")
print(f"OOF NMAE (Hurdle Log-L1)          : {nmae_score(y_train, oof_prob_spend * oof_reg_log):.4f}")
print(f"OOF NMAE (Tuned Hurdle Ensemble)  : {best_nmae:.4f} (alpha={alpha_opt:.2f}, gate={gate_opt:.3f}, scale={scale_opt:.3f})")

## 5. Save Diagnostics (`task3/assets/`) & Test Predictions (`task3/predictions/`)

In [ ]:
bench_t3 = pd.DataFrame([
    {"model": "Median Baseline", "mae": mean_absolute_error(y_train, np.full_like(y_train, np.median(y_train))), "nmae_score": nmae_score(y_train, np.full_like(y_train, np.median(y_train)))},
    {"model": "Direct L1 LightGBM", "mae": mean_absolute_error(y_train, oof_reg_raw_l1), "nmae_score": nmae_score(y_train, oof_reg_raw_l1)},
    {"model": "Two-Stage Hurdle (Prob * Log-L1)", "mae": mean_absolute_error(y_train, oof_prob_spend * oof_reg_log), "nmae_score": nmae_score(y_train, oof_prob_spend * oof_reg_log)},
    {"model": "Tuned Hurdle Ensemble", "mae": mean_absolute_error(y_train, oof_hurdle_final), "nmae_score": best_nmae},
])
bench_t3.to_csv(ASSETS_DIR / "model-benchmark.csv", index=False)
display(bench_t3.round(4))

fig, axes = plt.subplots(1, 2, figsize=(14, 5.2))
axes[0].scatter(np.log1p(y_train[:5000]), np.log1p(oof_hurdle_final[:5000]), alpha=0.25, s=12, color="#2563EB")
axes[0].plot([0, 13], [0, 13], "r--", lw=2, label="Perfect Prediction (y = y_hat)")
axes[0].set_title(f"Task 3 OOF Actual vs Predicted log(1+THB) | NMAE = {best_nmae:.4f}", fontsize=12, fontweight="bold")
axes[0].set_xlabel("Actual log(1 + spending_30d)")
axes[0].set_ylabel("Predicted log(1 + spending_30d)")
axes[0].legend()
axes[0].grid(alpha=0.3)

fi_df = pd.DataFrame({"feature": FEATURE_COLS, "importance": feature_importances}).sort_values("importance", ascending=False).head(15)
axes[1].barh(fi_df["feature"][::-1], fi_df["importance"][::-1], color="#D97706")
axes[1].set_title("Task 3: Top 15 Features for Monthly Spending", fontsize=12, fontweight="bold")
axes[1].set_xlabel("Average Split Importance (Stage 2 Log-L1)")
axes[1].grid(axis="x", alpha=0.3)

plt.tight_layout()
plt.savefig(ASSETS_DIR / "actual-vs-predicted-and-importance.png", dpi=300, bbox_inches="tight")
plt.show()

oof_out = pd.DataFrame({
    "player_id": train_df[ID_COL],
    "spending_true": y_train,
    "prob_spend": oof_prob_spend,
    "spending_pred": np.round(oof_hurdle_final, 2),
})
oof_out.to_csv(PREDICTIONS_DIR / "oof_predictions.csv", index=False)

test_out = pd.DataFrame({
    "player_id": test_df[ID_COL],
    "prob_spend": test_prob_spend,
    "task3": np.round(np.clip(test_hurdle_final, 0.0, 300000.0), 2),
})
test_out.to_csv(PREDICTIONS_DIR / "test_predictions.csv", index=False)

print(f"Saved Test predictions: {PREDICTIONS_DIR / 'test_predictions.csv'} ({len(test_out):,} rows)")
print(test_out["task3"].describe().round(2))